# Week 2 ID2221

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.dataframe import DataFrame
from pyspark.sql.window import Window
import pyspark.sql.functions as F

#The second line (with 4g) specifies how much RAM to use. change according to machine
spark = SparkSession.builder \
    .appName("IngestionFramework") \
    .config("spark.driver.memory", "4g") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

#AQE
#spark.conf.set("spark.sql.adaptive.enabled", "false")


26/09/20 20:26:09 WARN Utils: Your hostname, BabisPC resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/09/20 20:26:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/babis/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/babis/.ivy2/cache
The jars for the packages stored in: /home/babis/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-4ce836d9-74b0-4bd4-80ef-f99028351254;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 191ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

In [2]:
integrated_taxi_trips = spark.read.format("delta").load("delta/integrated_taxi_trips_by_borough")
integrated_taxi_trips_by_date = spark.read.format("delta").load("delta/integrated_taxi_trips_by_date")



In [3]:
#function to fairly measure time

import time

def measure(df, runs=3):
    """Run the query fully, several times, return the median time in ms."""
    times = []
    for i in range(runs):
        start = time.time()
        df.write.format("noop").mode("overwrite").save()   # forces full computation, writes nothing
        times.append((time.time() - start) * 1000)
    times.sort()
    return f"{round(times[len(times) // 2])} ms"   # median

1. Monthly taxi demand for each taxi zone

In [4]:
Q1 = (integrated_taxi_trips
          .select("pu_zone", "tpep_pickup_datetime")
          .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
          .groupBy(["pu_zone", "month"])
          .count())

# Uncomment this line if you are doing latency measurements
# By default, spark is lazy and only calculates when the results are requested

# result.collect()

#Q1.show(5)

measure(Q1)

'5700 ms'

In [5]:
#Q1.explain("formatted")

2. Average trip distance under different weather conditions

In [6]:
# Here we look at the average trip distance if there is rain, or no rain.

Q2 = (
    integrated_taxi_trips
          .select("trip_distance", "weather_prcp")
          .dropna()
          .withColumn("is_raining", F.col("weather_prcp") != 0.0)
          .groupBy("is_raining")
          .avg("trip_distance")
          .withColumn("avg_trip_distance", F.round("avg(trip_distance)", 2))
          .drop("weather_prcp", "avg(trip_distance)")
    )

measure(Q2)

'3494 ms'

In [7]:
Q2.show()

+----------+-----------------+
|is_raining|avg_trip_distance|
+----------+-----------------+
|      true|              4.6|
|     false|             4.68|
+----------+-----------------+



3. Relationship between air quality and taxi demand

In [8]:
# Get the hourly taxi demand associated to the average air quality measurement
Q3 = (
    integrated_taxi_trips
        .select("tpep_pickup_datetime","air_q_sample_measurement")
        # A bunch of lines do not have any air quality data, they are removed
        .dropna()
        .withColumn("hour", F.date_trunc("hour", "tpep_pickup_datetime"))
        .groupBy("hour")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.avg("air_q_sample_measurement").alias("air_quality")
        )
)

# Calculate the correlation between air quality and taxi demand
correlation = Q3.stat.corr(
    "air_quality",
    "taxi_demand"
)

print(correlation) # Result: corr = -0.017 -> No correlation 

-0.016985133728975584


In [9]:
measure(Q3)

'3086 ms'

4. Taxi zones with the largest variation in demand under different weather conditions

In [10]:
Q4 = (
    integrated_taxi_trips
        .select("pu_zone", "weather_prcp")
        .dropna()
        .withColumn("is_raining", F.col("weather_prcp") != 0.0)
        .groupBy("pu_zone", "is_raining")
        .agg(
            F.count("*").alias("taxi_demand")
        )
    )

Q4 = (
        Q4
        .select("pu_zone", "is_raining", "taxi_demand")
        .groupBy("pu_zone")
        .pivot("is_raining", [False, True])
        .sum("taxi_demand")
        .withColumnRenamed("false","no_rain_demand")
        .withColumnRenamed("true","rain_demand")
        .fillna(0)
        .withColumn("demand_variation", F.abs(F.col("rain_demand")-F.col("no_rain_demand")))
        .orderBy("demand_variation")
    )



Q4.show()

+--------------------+--------------+-----------+----------------+
|             pu_zone|no_rain_demand|rain_demand|demand_variation|
+--------------------+--------------+-----------+----------------+
|Governor's Island...|             0|          1|               1|
|   Rossville/Woodrow|             1|          0|               1|
|     Mariners Harbor|             2|          0|               2|
|       Rikers Island|             4|          6|               2|
|     Freshkills Park|             2|          0|               2|
|         Great Kills|             2|          0|               2|
|Eltingville/Annad...|             3|          0|               3|
|Charleston/Totten...|             4|          0|               4|
|         Westerleigh|             6|          2|               4|
|             Oakwood|             5|          0|               5|
|Heartland Village...|            12|          3|               9|
|Saint George/New ...|            11|          2|             

5. Peak travel hours for each day of the week

In [11]:
Q5 = (
    integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count()
)

# The window is used to get the max count for each day while keeping other informations (here, the busiest hour)
window = Window.partitionBy("day_of_week").orderBy(F.desc("count"))

Q5 = (
    Q5
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .drop("rank")
)


Q5.show()

+-----------+----+------+
|day_of_week|hour| count|
+-----------+----+------+
|     Friday|  18|110876|
|     Monday|  18| 87453|
|   Saturday|  19|103639|
|     Sunday|   0| 84325|
|   Thursday|  18|126855|
|    Tuesday|  18|106166|
|  Wednesday|  18|117895|
+-----------+----+------+



6. Monthly trends in taxi demand

In [12]:
Q6 = (integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy("month")
    .count()
 )

window = Window.orderBy("month")

# Calculate the difference of number of trips between the current month and the previous month
Q6 = (
    Q6
    .withColumn("previous_count", F.lag("count").over(window))
    .withColumn("difference", F.expr("count - previous_count"))
)

#Q6.show()

In [13]:
measure(Q6)

'3368 ms'

In [14]:
Q6.show()

+-------------------+-------+--------------+----------+
|              month|  count|previous_count|difference|
+-------------------+-------+--------------+----------+
|2002-12-01 00:00:00|      4|          NULL|      NULL|
|2008-12-01 00:00:00|      1|             4|        -3|
|2009-01-01 00:00:00|      4|             1|         3|
|2023-12-01 00:00:00|     10|             4|         6|
|2024-01-01 00:00:00|3073690|            10|   3073680|
|2024-02-01 00:00:00|3263634|       3073690|    189944|
|2024-03-01 00:00:00|3913076|       3263634|    649442|
|2024-04-01 00:00:00|      3|       3913076|  -3913073|
+-------------------+-------+--------------+----------+



## Caching

In [15]:

spark.catalog.clearCache()

print("Q1 no cached:", measure(Q1))
print("Q4 no cached:", measure(Q4))
print("Q5 no cached:", measure(Q5))
print("Q6 no cached:", measure(Q6))

base = integrated_taxi_trips.select("pu_zone", "tpep_pickup_datetime", "weather_prcp").cache()
base.count()  #action to force spark

Q1_cached = (base
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy(["pu_zone", "month"])
    .count())

window = Window.orderBy("month")
Q6_cached = (base
    .select("tpep_pickup_datetime")
    .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
    .groupBy("month")
    .count()
    .withColumn("previous_count", F.lag("count").over(window))
    .withColumn("difference", F.expr("count - previous_count")))

Q4_cached = (
    base
        .select("pu_zone", "weather_prcp")
        .dropna()
        .withColumn("is_raining", F.col("weather_prcp") != 0.0)
        .groupBy("pu_zone", "is_raining")
        .agg(
            F.count("*").alias("taxi_demand")
        )
    )

Q4_cached = (
        Q4_cached
        .select("pu_zone", "is_raining", "taxi_demand")
        .groupBy("pu_zone")
        .pivot("is_raining", [False, True])
        .sum("taxi_demand")
        .withColumnRenamed("false","no_rain_demand")
        .withColumnRenamed("true","rain_demand")
        .fillna(0)
        .withColumn("demand_variation", F.abs(F.col("rain_demand")-F.col("no_rain_demand")))
        .orderBy("demand_variation")
    )

window2 = Window.partitionBy("day_of_week").orderBy(F.desc("count"))
Q5_cached = (
    base
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count().withColumn("rank", F.row_number().over(window2))
    .filter(F.col("rank") == 1)
    .drop("rank")
)






#this should show "InMemoryTableScan" if the cache is actually being hit
#Q1_cached.explain("formatted")
#Q6_cached.explain("formatted")

print("Q1 cached:", measure(Q1_cached))
print("Q4 cached:", measure(Q4_cached))
print("Q5 cached:", measure(Q5_cached))
print("Q6 cached:", measure(Q6_cached))

spark.catalog.clearCache()

Q1 no cached: 3376 ms


Q4 no cached: 3231 ms


Q5 no cached: 3693 ms


Q6 no cached: 3272 ms


Q1 cached: 995 ms
Q4 cached: 666 ms


Q5 cached: 1152 ms
Q6 cached: 576 ms


## Partition Pruning

In [16]:
#average fare in Manhattan

integrated_taxi_trips = spark.read.format("delta").load("delta/integrated_taxi_trips_by_borough")

#prunes on the borough table
version_a = integrated_taxi_trips.filter(F.col("pu_borough") == "Manhattan").groupBy("pu_borough").agg(F.round(F.avg("fare_amount"), 4).alias("avg_fare"))

#prunes on time table
version_b = integrated_taxi_trips_by_date.filter(F.col("pu_borough") == "Manhattan").groupBy("pu_borough").agg(F.round(F.avg("fare_amount"), 4).alias("avg_fare"))

version_a.explain("formatted")
version_b.explain("formatted")


print("A (borough table):", measure(version_a))
print("B (time table):", measure(version_b))
# prove they're the same
assert version_a.exceptAll(version_b).count() == 0
assert version_b.exceptAll(version_a).count() == 0

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [fare_amount#6303, pu_borough#6337]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/delta/integrated_taxi_trips_by_borough]
PartitionFilters: [isnotnull(pu_borough#6337), (pu_borough#6337 = Manhattan)]
ReadSchema: struct<fare_amount:double>

(2) HashAggregate
Input [2]: [fare_amount#6303, pu_borough#6337]
Keys [1]: [pu_borough#6337]
Functions [1]: [partial_avg(fare_amount#6303)]
Aggregate Attributes [2]: [sum#6709, count#6710L]
Results [3]: [pu_borough#6337, sum#6711, count#6712L]

(3) Exchange
Input [3]: [pu_borough#6337, sum#6711, count#6712L]
Arguments: hashpartitioning(pu_borough#6337, 200), ENSURE_REQUIREMENTS, [plan_id=6516]

(4) HashAggregate
Input [3]: [pu_borough#6337, sum#6711, count#6712L]
Keys [1]: [pu_borough#6337]
Functions [1]: [avg(fare_amount#6303

== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [fare_amount#232, pu_borough#266, day_timestamp#293]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/delta/integrated_taxi_trips_by_date]
PushedFilters: [IsNotNull(pu_borough), EqualTo(pu_borough,Manhattan)]
ReadSchema: struct<fare_amount:double,pu_borough:string>

(2) Filter
Input [3]: [fare_amount#232, pu_borough#266, day_timestamp#293]
Condition : (isnotnull(pu_borough#266) AND (pu_borough#266 = Manhattan))

(3) Project
Output [2]: [fare_amount#232, pu_borough#266]
Input [3]: [fare_amount#232, pu_borough#266, day_timestamp#293]

(4) HashAggregate
Input [2]: [fare_amount#232, pu_borough#266]
Keys [1]: [pu_borough#266]
Functions [1]: [partial_avg(fare_amount#232)]
Aggregate Attributes [2]: [sum#7078, count#7

A (borough table): 1263 ms


B (time table): 26213 ms


## Broadcast joins vs. Shuffle joins

In [17]:
import pyspark.sql.functions as F

taxi_trips_all = (
    spark.read.format("delta").load("delta/taxi_trips_01")
    .unionByName(spark.read.format("delta").load("delta/taxi_trips_02"))
    .unionByName(spark.read.format("delta").load("delta/taxi_trips_03"))
)

weather = spark.read.format("delta").load("delta/weather")

#AQE can force broadcast. disable it
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")


# shuffle join
Q2_shuffle = (
    taxi_trips_all
    .withColumn("weather_time", F.date_trunc("hour", F.col("tpep_pickup_datetime")))
    .join(
        weather,
        on=F.col("weather_time") == F.col("timestamp"),
        how="inner"
    )
    .select("trip_distance", "prcp")
    .dropna(subset=["prcp"])
    .withColumn("is_raining", F.col("prcp") != 0.0)
    .groupBy("is_raining")
    .avg("trip_distance")
    .withColumn("avg_trip_distance", F.round("avg(trip_distance)", 2))
)

Q2_shuffle.explain(mode="formatted")
print("Shuffle Q2 Execution Time:", measure(Q2_shuffle))


# broadcast join
Q2_broadcast = (
    taxi_trips_all
    .withColumn("weather_time", F.date_trunc("hour", F.col("tpep_pickup_datetime")))
    .join(
        F.broadcast(weather), # The optimization hint
        on=F.col("weather_time") == F.col("timestamp"),
        how="inner"
    )
    .select("trip_distance", "prcp")
    .dropna(subset=["prcp"])
    .withColumn("is_raining", F.col("prcp") != 0.0)
    .groupBy("is_raining")
    .avg("trip_distance")
    .withColumn("avg_trip_distance", F.round("avg(trip_distance)", 2))
)

Q2_broadcast.explain(mode="formatted")
print("Broadcast Q2 Execution Time:", measure(Q2_broadcast))

# 3. Restore default Spark configurations 
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")

== Physical Plan ==
* Project (26)
+- * HashAggregate (25)
   +- Exchange (24)
      +- * HashAggregate (23)
         +- * Project (22)
            +- * SortMergeJoin Inner (21)
               :- * Sort (15)
               :  +- Exchange (14)
               :     +- Union (13)
               :        :- * Project (4)
               :        :  +- * Filter (3)
               :        :     +- * ColumnarToRow (2)
               :        :        +- Scan parquet  (1)
               :        :- * Project (8)
               :        :  +- * Filter (7)
               :        :     +- * ColumnarToRow (6)
               :        :        +- Scan parquet  (5)
               :        +- * Project (12)
               :           +- * Filter (11)
               :              +- * ColumnarToRow (10)
               :                 +- Scan parquet  (9)
               +- * Sort (20)
                  +- Exchange (19)
                     +- * Filter (18)
                        +- * ColumnarToRow 

Shuffle Q2 Execution Time: 5148 ms
== Physical Plan ==
* Project (23)
+- * HashAggregate (22)
   +- Exchange (21)
      +- * HashAggregate (20)
         +- * Project (19)
            +- * BroadcastHashJoin Inner BuildRight (18)
               :- Union (13)
               :  :- * Project (4)
               :  :  +- * Filter (3)
               :  :     +- * ColumnarToRow (2)
               :  :        +- Scan parquet  (1)
               :  :- * Project (8)
               :  :  +- * Filter (7)
               :  :     +- * ColumnarToRow (6)
               :  :        +- Scan parquet  (5)
               :  +- * Project (12)
               :     +- * Filter (11)
               :        +- * ColumnarToRow (10)
               :           +- Scan parquet  (9)
               +- BroadcastExchange (17)
                  +- * Filter (16)
                     +- * ColumnarToRow (15)
                        +- Scan parquet  (14)


(1) Scan parquet 
Output [2]: [tpep_pickup_datetime#8146, trip_distanc

Broadcast Q2 Execution Time: 4227 ms


## AQE: on vs off

In [18]:
#AQE on
Q5_on = (
    integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count()
)

# The window is used to get the max count for each day while keeping other informations (here, the busiest hour)
window = Window.partitionBy("day_of_week").orderBy(F.desc("count"))

Q5_on = (
    Q5_on
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

print("Q5 execution time with AQE on:", measure(Q5_on))
Q5_on.explain("formatted")


Q5 execution time with AQE on: 3861 ms
== Physical Plan ==
AdaptiveSparkPlan (14)
+- Project (13)
   +- Filter (12)
      +- Window (11)
         +- WindowGroupLimit (10)
            +- Sort (9)
               +- Exchange (8)
                  +- WindowGroupLimit (7)
                     +- Sort (6)
                        +- HashAggregate (5)
                           +- Exchange (4)
                              +- HashAggregate (3)
                                 +- Project (2)
                                    +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [tpep_pickup_datetime#6294, pu_borough#6337]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/delta/integrated_taxi_trips_by_borough]
ReadSchema: struct<tpep_pickup_datetime:timestamp>

(2) Project
Output [2]: [hour(tpep_pickup_datetime#6294, Some(Europe/Stockholm)) AS hour#14424, date_format(tpep_pickup_datetime#6294, EEEE, Some(Europe/Stockholm)) AS day_of_week#144

In [19]:
#AQE off
spark.conf.set("spark.sql.adaptive.enabled", "false")

Q5_off = (
    integrated_taxi_trips
    .select("tpep_pickup_datetime")
    .withColumn("hour", F.hour("tpep_pickup_datetime"))
    .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
    .groupBy("day_of_week", "hour")
    .count()
)

# The window is used to get the max count for each day while keeping other informations (here, the busiest hour)
window = Window.partitionBy("day_of_week").orderBy(F.desc("count"))

Q5_off = (
    Q5_off
    .withColumn("rank", F.row_number().over(window))
    .filter(F.col("rank") == 1)
    .drop("rank")
)

print("Q5 execution time with AQE off:", measure(Q5_off))
Q5_off.explain("formatted")


spark.conf.set("spark.sql.adaptive.enabled", "true")

Q5 execution time with AQE off: 4086 ms
== Physical Plan ==
* Project (14)
+- * Filter (13)
   +- Window (12)
      +- WindowGroupLimit (11)
         +- * Sort (10)
            +- Exchange (9)
               +- WindowGroupLimit (8)
                  +- * Sort (7)
                     +- * HashAggregate (6)
                        +- Exchange (5)
                           +- * HashAggregate (4)
                              +- * Project (3)
                                 +- * ColumnarToRow (2)
                                    +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [tpep_pickup_datetime#6294, pu_borough#6337]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/delta/integrated_taxi_trips_by_borough]
ReadSchema: struct<tpep_pickup_datetime:timestamp>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [tpep_pickup_datetime#6294, pu_borough#6337]

(3) Project [codegen id : 1]
Output [2]: [hour(tpep_pickup_datetime#6294, Some

### Making analytical data objects

In [20]:

def write_table(df, table_name):
    
    output_path = f"./delta/assignment4_task4/{table_name}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(output_path)
    

def make_analytic_data_obj():
        # Taxi zone statistics
    source = f"./delta/integrated_taxi_trips_by_borough"
    Q1_local = (integrated_taxi_trips
              .select("pu_zone", "tpep_pickup_datetime", "trip_distance", "fare_amount")
              .withColumn("month", F.date_trunc("month", "tpep_pickup_datetime"))
               .groupby(["pu_zone", "month"])
              .agg(
                    F.count("*").alias("taxi_demand"),
                    F.round(F.avg("trip_distance")).alias("avg_trip_distance"),
                    F.round(F.sum("trip_distance")).alias("total_trip_distance"),
                    F.round(F.avg("fare_amount")).alias("avg_fare_amount"),
                    F.round(F.sum("fare_amount")).alias("total_fare_amount")
              )
              .withColumn("data_source", F.lit(source)) 
              .withColumn("creation_time", F.current_timestamp())
              .withColumn("refresh_time", F.current_timestamp())
              .withColumn("schema_version", F.lit("1.0"))
            )
    
    write_table(Q1_local, table_name = "taxi_zone_statistics")

    # weather impact summary
    Q2_local = (integrated_taxi_trips
          .select("trip_distance", "weather_prcp")
          .dropna()
          .withColumn("is_raining", F.col("weather_prcp") != 0.0)
          .groupBy("is_raining")
          .agg(
            F.count("*").alias("taxi_demand"),
            F.round(F.avg("trip_distance"),2).alias("avg_trip_distance"),
            F.round(F.sum("trip_distance")).alias("total_trip_distance")       
          )
          .withColumn("data_source", F.lit(source)) 
          .withColumn("creation_time", F.current_timestamp())
          .withColumn("refresh_time", F.current_timestamp())
          .withColumn("schema_version", F.lit("1.0"))
    )

    
    write_table(Q2_local, table_name = "weather_impact_summary")

    # Air Quality Impact Summary 
    Q3_local = (
    integrated_taxi_trips
        .select("tpep_pickup_datetime","air_q_sample_measurement", "trip_distance")
        # A bunch of lines do not have any air quality data, they are removed
        .dropna(subset = ["tpep_pickup_datetime", "air_q_sample_measurement"])
        .withColumn("pickup_hour", F.date_trunc("hour", "tpep_pickup_datetime"))
        .groupBy("pickup_hour")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.avg("air_q_sample_measurement").alias("avg_air_quality"),
            F.round(F.avg("trip_distance")).alias("avg_trip_distance"),
            F.round(F.sum("trip_distance")).alias("total_trip_distance")
        )
        .withColumn("data_source", F.lit(source)) 
        .withColumn("creation_time", F.current_timestamp())
        .withColumn("refresh_time", F.current_timestamp())
        .withColumn("schema_version", F.lit("1.0"))
    )

    write_table(Q3_local, table_name = "air_quality_impact_summary")
    
    # Daily Mobility Summary
    Q5_local = (
        integrated_taxi_trips
        .select("tpep_pickup_datetime", "trip_distance")
        .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
        .withColumn("day_of_week", F.date_format("tpep_pickup_datetime", "EEEE"))
        .groupBy("pickup_hour", "day_of_week")
        .agg(
            F.count("*").alias("taxi_demand"),
            F.round(F.avg("trip_distance")).alias("avg_trip_distance"),
            F.round(F.sum("trip_distance")).alias("total_trip_distance")
            
        )
        .withColumn("data_source", F.lit(source)) 
        .withColumn("creation_time", F.current_timestamp())
        .withColumn("refresh_time", F.current_timestamp())
        .withColumn("schema_version", F.lit("1.0")).orderBy("day_of_week","pickup_hour")
    )

    write_table(Q5_local, table_name = "daily_mobility_summary")
    


In [21]:
make_analytic_data_obj()

In [22]:
taxi_zone_statistics = spark.read.format("delta").load("delta/assignment4_task4/taxi_zone_statistics")
# taxi_zone_statistics.show()
taxi_zone_statistics.limit(20).toPandas()

,pu_zone,month,taxi_demand,avg_trip_distance,total_trip_distance,avg_fare_amount,total_fare_amount,data_source,creation_time,refresh_time,schema_version
0,Upper West Side South,2024-03-01,102427,2.0,222993.0,14.0,1433719.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
1,Washington Heights North,2024-01-01,495,5.0,2345.0,29.0,14428.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
2,SoHo,2024-03-01,28612,2.0,68447.0,16.0,463141.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
3,Yorkville East,2024-03-01,48901,8.0,403163.0,15.0,718369.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
4,Upper West Side South,2024-02-01,90849,6.0,507412.0,14.0,1244708.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
5,Yorkville West,2024-02-01,59468,6.0,348715.0,13.0,781514.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
6,Central Park,2024-03-01,53445,2.0,117877.0,15.0,797913.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
7,East Chelsea,2024-02-01,80508,2.0,197378.0,16.0,1320275.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
8,Lenox Hill West,2024-03-01,83088,2.0,164325.0,13.0,1095384.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0
9,Manhattan Valley,2024-03-01,28312,4.0,114511.0,15.0,419258.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:31:58.308834,2026-09-20 20:31:58.308834,1.0


In [23]:
air_qual_summary = spark.read.format("delta").load("delta/assignment4_task4/air_quality_impact_summary")
# taxi_zone_statistics.show()
air_qual_summary.limit(20).toPandas()

,pickup_hour,taxi_demand,avg_air_quality,avg_trip_distance,total_trip_distance,data_source,creation_time,refresh_time,schema_version
0,2024-01-29 04:00:00,96,2.806250,8.0,782.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
1,2024-02-04 21:00:00,1282,7.179407,14.0,17385.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
2,2024-03-05 09:00:00,840,1.719048,10.0,8751.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
3,2024-03-06 10:00:00,810,25.250000,10.0,7971.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
4,2024-03-14 22:00:00,1142,27.588091,13.0,14470.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
5,2024-03-06 21:00:00,1126,3.849645,12.0,13640.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
6,2024-01-19 12:00:00,690,13.753478,11.0,7894.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
7,2024-02-08 21:00:00,1094,20.940311,12.0,13392.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
8,2024-02-27 18:00:00,1066,13.312383,12.0,12362.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0
9,2024-01-29 09:00:00,862,5.897912,11.0,9672.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:15.052367,2026-09-20 20:32:15.052367,1.0


In [24]:
weather_summary = spark.read.format("delta").load("delta/assignment4_task4/weather_impact_summary")
# taxi_zone_statistics.show()
weather_summary.limit(20).toPandas()

,is_raining,taxi_demand,avg_trip_distance,total_trip_distance,data_source,creation_time,refresh_time,schema_version
0,True,1161506,4.60,5341197.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:07.960762,2026-09-20 20:32:07.960762,1.0
1,False,8320507,4.68,38944876.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:07.960762,2026-09-20 20:32:07.960762,1.0


In [25]:
daily_mobility = spark.read.format("delta").load("delta/assignment4_task4/daily_mobility_summary")
# taxi_zone_statistics.show()
daily_mobility.limit(20).toPandas()


,pickup_hour,day_of_week,taxi_demand,avg_trip_distance,total_trip_distance,data_source,creation_time,refresh_time,schema_version
0,0,Friday,40166,4.0,176003.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
1,1,Friday,20355,4.0,78127.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
2,2,Friday,10963,3.0,37104.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
3,3,Friday,7197,4.0,28047.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
4,4,Friday,7085,5.0,37585.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
5,5,Friday,11096,27.0,298319.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
6,6,Friday,24117,10.0,236572.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
7,7,Friday,46252,12.0,549573.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
8,8,Friday,60968,5.0,302444.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0
9,9,Friday,64109,5.0,329667.0,./delta/integrated_taxi_trips_by_borough,2026-09-20 20:32:22.591864,2026-09-20 20:32:22.591864,1.0


### Evaluate platform

In [26]:
# Code for querying the task 4 tables (before any tassk 3 optimization) 
def q1_task4(table, extra_flag = False):
    results = (table.select("pu_zone","month","taxi_demand"))
    if (extra_flag == True):           
        return results.collect() # maybe show
    else:
        return results

def q2_task4(table, extra_flag = False):
    results = (table.select("is_raining","avg_trip_distance"))
    if extra_flag == True:           
        return results.collect(), corr # maybe show
    else:
        return results



def q3_task4(table, extra_flag=False):
    results = (table.select("pickup_hour","taxi_demand", "avg_air_quality"))
    corr = results.stat.corr("taxi_demand", "avg_air_quality") 
    print(corr)
    if extra_flag == True:           
        return results.collect(), corr # maybe show
    else:
        return results

def q5_task4(table, extra_flag=False):
    window = (Window
             .partitionBy("day_of_week")
             .orderBy(F.desc("taxi_demand")))
    results = (table.select("day_of_week", "pickup_hour", "taxi_demand")
              .withColumn("rank", F.row_number().over(window)) 
              .filter(F.col("rank") == 1)
              .drop("rank")) 
    if (extra_flag == True):           
        return results.collect(), corr # maybe show
    else:
        return results

def q6_task4(table, extra_flag=False):
    q6_temp = (table
    .groupBy("month")
    .agg(F.sum("taxi_demand").alias("count")) # forgot to give correct name before so keep it as count here
     )
   

    window = Window.orderBy("month")
    
    # Calculate the difference of number of trips between the current month and the previous month
    results = (
        q6_temp
        .withColumn("previous_count", F.lag("count").over(window))
        .withColumn("difference", F.expr("count - previous_count"))
    )
    if (extra_flag == True):   
        return results.collect() # maybe show
    else:
        return results

def get_explain_format(df):
    df.explain(mode = "formatted")

def compare_qs(qa, qb):
    if( qa.exceptAll(qb).count() == 0 and qb.exceptAll(qa).count() == 0):
        print("the two queries give the same results")
    else:
        print("The two queries give different results")
        print(f"extra lines in qa: {qa.exceptAll(qb).count() == 0 } and extra lines in qb: {qb.exceptAll(qa).count() == 0}")


In [33]:
# task 4 version (no task 3 optimization):

get_explain_format(Q1)

get_explain_format(Q5)

get_explain_format(Q6)

print("query q1")
measure(Q1)
print("query q5")
measure(Q5)
print("query q6")
measure(Q6)
# task 3 version (including task 4 twist):
# def data_prod_normal():
Q1_prod = q1_task4(taxi_zone_statistics)
# Q2_prod  = q2_task4(weather_summary)
Q5_prod = q5_task4(daily_mobility)
Q6_prod = q6_task4(taxi_zone_statistics)

get_explain_format(Q1_prod)
get_explain_format(Q5_prod)
get_explain_format(Q6_prod)
print(f"Execution time Q1_prod (no task 3 optimization): {measure(Q1_prod)}")
# print(f"Execution time Q2_prod (no task 3 optimization): {measure(Q2_prod)}")
print(f"Execution time Q5_prod (no task 3 optimization): {measure(Q5_prod)}")

print(f"Execution time Q5_prod (no task 3 optimization): {measure(Q6_prod)}")

    # return Q1_prod, Q5_prod, Q6_prod


    
# def cache_data_prod():   
spark.catalog.clearCache()

tzs_cached = taxi_zone_statistics.cache()
tzs_cached.count()  #action to force spark

# aqs_cached = air_qual_summary.cache()
# aqs_cached.count()

dm_cached = daily_mobility.cache()
dm_cached.count()

get_explain_format(Q1_prod)
Q1_cached_task5 = q1_task4(tzs_cached)
# Q2_cached  = q2_task4(aqs_cached)
Q5_cached_task5 = q5_task4(dm_cached)
Q6_cached_task5 = q6_task4(tzs_cached)





get_explain_format(Q1_cached_task5)


get_explain_format(Q5_cached_task5)


get_explain_format(Q6)

get_explain_format(Q6_cached_task5)

print("Q1 prod cached:", measure(Q1_cached_task5))
# print("Q4 cached:", measure(Q4_cached))
print("Q5 prod cached:", measure(Q5_cached_task5))
print("Q6 prod cached:", measure(Q6_cached_task5))

compare_qs(Q1, Q1_prod)
compare_qs(Q1, Q1_cached_task5)
compare_qs(Q5, Q5_prod)
compare_qs(Q5, Q5_cached_task5)
compare_qs(Q6, Q6_prod)
compare_qs(Q6, Q6_cached_task5)

spark.catalog.clearCache()

# return Q1_cached, Q5_cached, Q6_cached
    
    
    

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [tpep_pickup_datetime#49, pu_zone#93, pu_borough#92]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/delta/integrated_taxi_trips_by_borough]
ReadSchema: struct<tpep_pickup_datetime:timestamp,pu_zone:string>

(2) Project
Output [2]: [pu_zone#93, date_trunc(month, tpep_pickup_datetime#49, Some(Europe/Stockholm)) AS month#369]
Input [3]: [tpep_pickup_datetime#49, pu_zone#93, pu_borough#92]

(3) HashAggregate
Input [2]: [pu_zone#93, month#369]
Keys [2]: [pu_zone#93, month#369]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#384L]
Results [3]: [pu_zone#93, month#369, count#385L]

(4) Exchange
Input [3]: [pu_zone#93, month#369, count#385L]
Arguments: hashpartitioning(pu_zone#93, month#369, 200), ENSURE_REQUIREMENTS, [plan_id

query q5


query q6


== Physical Plan ==
* ColumnarToRow (2)
+- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pu_zone#20703, month#20704, taxi_demand#20705L]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/delta/assignment4_task4/taxi_zone_statistics]
ReadSchema: struct<pu_zone:string,month:timestamp,taxi_demand:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [pu_zone#20703, month#20704, taxi_demand#20705L]


== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- Filter (9)
      +- Window (8)
         +- WindowGroupLimit (7)
            +- Sort (6)
               +- Exchange (5)
                  +- WindowGroupLimit (4)
                     +- Sort (3)
                        +- Project (2)
                           +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_hour#23535, day_of_week#23536, taxi_demand#23537L]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/d

the two queries give the same results


the two queries give the same results


the two queries give the same results


the two queries give the same results


the two queries give the same results


the two queries give the same results


In [29]:

#AQE off
spark.conf.set("spark.sql.adaptive.enabled", "false")

Q5_off_task5 = q5_task4(daily_mobility)

Q5_off_task5.explain("formatted")

print("Q5 on product execution time with AQE off:", measure(Q5_off_task5))

spark.conf.set("spark.sql.adaptive.enabled", "true")
#AQE on
Q5_on_task5 = q5_task4(daily_mobility)

Q5_on_task5.explain("formatted")

print("Q5 on product execution time with AQE on:", measure(Q5_on_task5))

compare_qs(Q5, Q5_on_task5)
compare_qs(Q5, Q5_off_task5)



== Physical Plan ==
* Project (11)
+- * Filter (10)
   +- Window (9)
      +- WindowGroupLimit (8)
         +- * Sort (7)
            +- Exchange (6)
               +- WindowGroupLimit (5)
                  +- * Sort (4)
                     +- * Project (3)
                        +- * ColumnarToRow (2)
                           +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [pickup_hour#23535, day_of_week#23536, taxi_demand#23537L]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/delta/assignment4_task4/daily_mobility_summary]
ReadSchema: struct<pickup_hour:int,day_of_week:string,taxi_demand:bigint>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [pickup_hour#23535, day_of_week#23536, taxi_demand#23537L]

(3) Project [codegen id : 1]
Output [3]: [day_of_week#23536, pickup_hour#23535, taxi_demand#23537L]
Input [3]: [pickup_hour#23535, day_of_week#23536, taxi_demand#23537L]

(4) Sort [codegen id : 1]
Input [3]: [day_of_week#23

the two queries give the same results


the two queries give the same results


In [30]:
# Additional comparisons for the sake of completion

Q2_prod = q2_task4(weather_summary)
Q3_prod = q3_task4(air_qual_summary)
get_explain_format(Q2)
get_explain_format(Q3)
get_explain_format(Q2_prod)
get_explain_format(Q3_prod)
print("Q2", measure(Q2))
print("Q3", measure(Q3))
print("Q2_prod", measure(Q2_prod))
print("Q3_prod", measure(Q3_prod))
compare_qs(Q2,Q2_prod)
compare_qs(Q3,Q3_prod)

-0.016985133728975584
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [trip_distance#52, weather_prcp#76, pu_borough#92]
Batched: true
Location: PreparedDeltaFileIndex [file:/mnt/c/Users/Babis/desktop/ID2221/lab2/ID2221_Lab/delta/integrated_taxi_trips_by_borough]
ReadSchema: struct<trip_distance:double,weather_prcp:double>

(2) Filter
Input [3]: [trip_distance#52, weather_prcp#76, pu_borough#92]
Condition : atleastnnonnulls(2, trip_distance#52, weather_prcp#76)

(3) Project
Output [2]: [trip_distance#52, NOT (weather_prcp#76 = 0.0) AS is_raining#1480]
Input [3]: [trip_distance#52, weather_prcp#76, pu_borough#92]

(4) HashAggregate
Input [2]: [trip_distance#52, is_raining#1480]
Keys [1]: [is_raining#1480]
Functions [1]: [partial_avg(trip_distance#52)]
Aggregate Attributes [2]: [sum#1499, count#1500L]
Results [3

Q2 2249 ms


Q3 3018 ms
Q2_prod 1410 ms
Q3_prod 1449 ms


the two queries give the same results


the two queries give the same results
